In [ ]:
!pip -q install duckdb pyarrow pandas numpy tqdm sentence-transformers psutil

In [ ]:
import os
import json
import time
import shutil
import glob
import math
import re
import psutil
import duckdb
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import torch
import gc

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print(str(e))

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output"
INPUT_PATH = os.path.join(BASE_DIR, "san_pham.parquet")
WORK_DIR = "/content/semantic_search_full_mpnet_cosine_eval"
EXPORT_DIR = os.path.join(BASE_DIR, "semantic_search_full_mpnet_cosine_eval")

CLEAN_PARQUET_PATH = os.path.join(WORK_DIR, "clean_products.parquet")
EMBEDDING_DIR = os.path.join(WORK_DIR, "embedding_shards")
METADATA_DIR = os.path.join(WORK_DIR, "metadata_shards")
METADATA_DB_PATH = os.path.join(WORK_DIR, "metadata.duckdb")
ETL_STATS_PATH = os.path.join(WORK_DIR, "etl_stats.json")
RUN_STATS_PATH = os.path.join(WORK_DIR, "run_stats.json")
EVAL_METRICS_PATH = os.path.join(WORK_DIR, "category_evaluation_metrics.csv")
EVAL_SUMMARY_PATH = os.path.join(WORK_DIR, "category_evaluation_summary.csv")

MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"
MAX_PRODUCTS = None
EMBED_BATCH_ROWS = 30000
ENCODE_BATCH_SIZE = 256
MAX_SEQ_LENGTH = 128
EVAL_QUERY_COUNT = 200
EVAL_K = 10
EVAL_QUERY_BATCH_SIZE = 32

RESET_WORK_DIR = False
LOAD_EXISTING_EXPORT = True
FORCE_REBUILD_CLEAN = False
FORCE_REBUILD_EMBEDDINGS = False
FORCE_REBUILD_METADATA_DB = False
FORCE_REBUILD_EVAL = False
SAVE_TO_DRIVE = True
USE_FP16_ON_GPU = True
USE_GPU_FOR_BATCH_RETRIEVAL = True

if LOAD_EXISTING_EXPORT and os.path.exists(EXPORT_DIR) and not os.path.exists(WORK_DIR):
    shutil.copytree(EXPORT_DIR, WORK_DIR)

if RESET_WORK_DIR and os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(EMBEDDING_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print(WORK_DIR)
print(EXPORT_DIR)

/content/semantic_search_full_mpnet_cosine_eval
/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/semantic_search_full_mpnet_cosine_eval


In [ ]:
def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

def qident(value):
    return '"' + str(value).replace('"', '""') + '"'

def duck_parquet_path(path):
    if os.path.isdir(path):
        files = []
        for root, _, names in os.walk(path):
            for name in names:
                if name.endswith(".parquet"):
                    files.append(os.path.join(root, name))
        if len(files) == 1:
            return files[0]
        return os.path.join(path, "**", "*.parquet")
    return path

def pick_column(columns, names):
    lower_map = {c.lower(): c for c in columns}
    for name in names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    for c in columns:
        cl = c.lower()
        for name in names:
            if name.lower() in cl:
                return c
    return None

def duck_text_expr(col, max_len):
    if col is None:
        return "''"
    x = "CAST(" + qident(col) + " AS VARCHAR)"
    x = "COALESCE(" + x + ", '')"
    x = "regexp_replace(" + x + ", '\\\\s+', ' ', 'g')"
    x = "trim(" + x + ")"
    if max_len is not None:
        x = "substr(" + x + ", 1, " + str(max_len) + ")"
    return x

def normalize_key_expr(expr):
    return "trim(regexp_replace(lower(" + expr + "), '[^a-z0-9]+', ' ', 'g'))"

def dcg_at_k(rels, k):
    rels = np.asarray(rels[:k], dtype=float)
    if len(rels) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(rels) + 2))
    return float(np.sum(rels / discounts))

def ndcg_at_k(rels, k):
    dcg = dcg_at_k(rels, k)
    ideal = sorted(rels, reverse=True)
    idcg = dcg_at_k(ideal, k)
    if idcg == 0:
        return 0.0
    return float(dcg / idcg)

def mrr_at_k(rels, k):
    for i, r in enumerate(rels[:k], start=1):
        if r > 0:
            return float(1.0 / i)
    return 0.0

In [ ]:
source_dataset = ds.dataset(INPUT_PATH, format="parquet")
ALL_COLUMNS = source_dataset.schema.names

ASIN_COL = pick_column(ALL_COLUMNS, ["asin", "parent_asin", "product_id", "item_id"])
TITLE_COL = pick_column(ALL_COLUMNS, ["title", "name", "product_name"])
BRAND_COL = pick_column(ALL_COLUMNS, ["brand", "store", "manufacturer"])
CATEGORY_COL = pick_column(ALL_COLUMNS, ["main_category", "category", "categories", "bc"])
DESCRIPTION_COL = pick_column(ALL_COLUMNS, ["description", "desc", "about_product"])
FEATURES_COL = pick_column(ALL_COLUMNS, ["features", "feature", "bullets"])
DETAILS_COL = pick_column(ALL_COLUMNS, ["details", "specs", "specifications"])
PRICE_COL = pick_column(ALL_COLUMNS, ["price", "actual_price", "discounted_price"])

required = {"asin": ASIN_COL, "title": TITLE_COL}
missing = [k for k, v in required.items() if v is None]
if missing:
    raise ValueError("missing required columns: " + str(missing))

column_map = {
    "asin": ASIN_COL,
    "title": TITLE_COL,
    "brand": BRAND_COL,
    "category": CATEGORY_COL,
    "description": DESCRIPTION_COL,
    "features": FEATURES_COL,
    "details": DETAILS_COL,
    "price": PRICE_COL
}

print(json.dumps(column_map, ensure_ascii=False, indent=2))

{
  "asin": "asin",
  "title": "title",
  "brand": "store",
  "category": "main_category",
  "description": "description",
  "features": "features",
  "details": "details",
  "price": "price"
}


In [ ]:
etl_start = time.time()
input_for_duckdb = duck_parquet_path(INPUT_PATH)

if os.path.exists(CLEAN_PARQUET_PATH) and not FORCE_REBUILD_CLEAN:
    con = duckdb.connect()
    raw_count = None
    clean_count = con.execute("SELECT count(*) FROM read_parquet(" + sql_literal(CLEAN_PARQUET_PATH) + ")").fetchone()[0]
    con.close()
    etl_stats = {
        "skipped": True,
        "input_path": INPUT_PATH,
        "clean_parquet_path": CLEAN_PARQUET_PATH,
        "raw_count": raw_count,
        "clean_count": int(clean_count),
        "elapsed_seconds": 0
    }
else:
    mem_gb = max(4, int(psutil.virtual_memory().total / 1024**3 * 0.7))
    threads = max(2, os.cpu_count() or 2)
    con = duckdb.connect()
    con.execute("PRAGMA threads=" + str(threads))
    con.execute("PRAGMA memory_limit='" + str(mem_gb) + "GB'")
    con.execute("PRAGMA temp_directory='/content/duckdb_tmp'")

    raw_count = con.execute("SELECT count(*) FROM read_parquet(" + sql_literal(input_for_duckdb) + ")").fetchone()[0]

    asin_expr = duck_text_expr(ASIN_COL, 120)
    title_expr = duck_text_expr(TITLE_COL, 400)
    brand_expr = duck_text_expr(BRAND_COL, 180)
    category_expr = duck_text_expr(CATEGORY_COL, 700)
    description_expr = duck_text_expr(DESCRIPTION_COL, 1200)
    features_expr = duck_text_expr(FEATURES_COL, 1200)
    details_expr = duck_text_expr(DETAILS_COL, 1200)
    price_expr = duck_text_expr(PRICE_COL, 120)

    retrieval_expr = "lower(trim(regexp_replace(concat_ws(' ', title, brand, category, description, features, details), '\\s+', ' ', 'g')))"
    category_key_expr = normalize_key_expr("category")

    limit_clause = "" if MAX_PRODUCTS is None else " LIMIT " + str(int(MAX_PRODUCTS))

    query = f"""
    COPY (
        WITH base AS (
            SELECT
                {asin_expr} AS asin,
                {title_expr} AS title,
                {brand_expr} AS brand,
                {category_expr} AS category,
                {description_expr} AS description,
                {features_expr} AS features,
                {details_expr} AS details,
                {price_expr} AS price
            FROM read_parquet({sql_literal(input_for_duckdb)})
        ),
        clean AS (
            SELECT
                row_number() OVER () - 1 AS row_id,
                asin,
                title,
                brand,
                category,
                {category_key_expr} AS category_key,
                price,
                {retrieval_expr} AS retrieval_text
            FROM base
            WHERE length(asin) > 0 AND length(title) > 0
            {limit_clause}
        )
        SELECT *
        FROM clean
        WHERE length(retrieval_text) > 10
    ) TO {sql_literal(CLEAN_PARQUET_PATH)} (FORMAT PARQUET, COMPRESSION SNAPPY)
    """

    con.execute(query)
    clean_count = con.execute("SELECT count(*) FROM read_parquet(" + sql_literal(CLEAN_PARQUET_PATH) + ")").fetchone()[0]
    con.close()

    etl_stats = {
        "skipped": False,
        "input_path": INPUT_PATH,
        "clean_parquet_path": CLEAN_PARQUET_PATH,
        "raw_count": int(raw_count),
        "clean_count": int(clean_count),
        "column_map": column_map,
        "elapsed_seconds": round(time.time() - etl_start, 2)
    }

with open(ETL_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(etl_stats, f, ensure_ascii=False, indent=2)

print(json.dumps(etl_stats, ensure_ascii=False, indent=2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "skipped": false,
  "input_path": "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/san_pham.parquet",
  "clean_parquet_path": "/content/semantic_search_full_mpnet_cosine_eval/clean_products.parquet",
  "raw_count": 2410915,
  "clean_count": 2410765,
  "column_map": {
    "asin": "asin",
    "title": "title",
    "brand": "store",
    "category": "main_category",
    "description": "description",
    "features": "features",
    "details": "details",
    "price": "price"
  },
  "elapsed_seconds": 519.33
}


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.cuda.empty_cache()

model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = MAX_SEQ_LENGTH

if device == "cuda" and USE_FP16_ON_GPU:
    try:
        model._first_module().auto_model.half()
    except Exception as e:
        print(str(e))

embedding_dim = model.get_sentence_embedding_dimension()

print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print(MODEL_NAME)
print(embedding_dim)
print(ENCODE_BATCH_SIZE)
print(MAX_SEQ_LENGTH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

cuda
Tesla T4
paraphrase-multilingual-mpnet-base-v2
768
256
128


/tmp/ipykernel_13809/2454769796.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = model.get_sentence_embedding_dimension()


In [ ]:
embedding_start = time.time()

existing_embedding_files = sorted(glob.glob(os.path.join(EMBEDDING_DIR, "embeddings_*.npy")))
existing_metadata_files = sorted(glob.glob(os.path.join(METADATA_DIR, "metadata_*.parquet")))

if existing_embedding_files and existing_metadata_files and not FORCE_REBUILD_EMBEDDINGS:
    total_vectors = 0
    for path in tqdm(existing_embedding_files):
        total_vectors += int(np.load(path, mmap_mode="r").shape[0])
    embedding_stats = {
        "skipped": True,
        "embedding_dir": EMBEDDING_DIR,
        "metadata_dir": METADATA_DIR,
        "n_products_embedded": int(total_vectors),
        "n_shards": int(len(existing_embedding_files)),
        "embedding_dim": int(np.load(existing_embedding_files[0], mmap_mode="r").shape[1]),
        "elapsed_seconds": 0
    }
else:
    for p in [EMBEDDING_DIR, METADATA_DIR]:
        if os.path.exists(p):
            shutil.rmtree(p)
        os.makedirs(p, exist_ok=True)

    clean_dataset = ds.dataset(CLEAN_PARQUET_PATH, format="parquet")
    shard_id = 0
    total_rows = 0
    columns = ["row_id", "asin", "title", "brand", "category", "category_key", "price", "retrieval_text"]

    for batch in clean_dataset.to_batches(batch_size=EMBED_BATCH_ROWS, columns=columns):
        df = pa.Table.from_batches([batch]).to_pandas()
        if len(df) == 0:
            continue

        texts = df["retrieval_text"].fillna("").astype(str).tolist()

        embeddings = model.encode(
            texts,
            batch_size=ENCODE_BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True
        ).astype("float32")

        embedding_path = os.path.join(EMBEDDING_DIR, "embeddings_{:06d}.npy".format(shard_id))
        metadata_path = os.path.join(METADATA_DIR, "metadata_{:06d}.parquet".format(shard_id))

        np.save(embedding_path, embeddings)

        meta = df[["row_id", "asin", "title", "brand", "category", "category_key", "price"]].copy()
        meta["row_id"] = meta["row_id"].astype(np.int64)
        pq.write_table(pa.Table.from_pandas(meta, preserve_index=False), metadata_path, compression="snappy")

        total_rows += int(len(df))
        shard_id += 1

        del df, texts, embeddings, meta
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    embedding_stats = {
        "skipped": False,
        "embedding_dir": EMBEDDING_DIR,
        "metadata_dir": METADATA_DIR,
        "n_products_embedded": int(total_rows),
        "n_shards": int(shard_id),
        "embedding_dim": int(embedding_dim),
        "model_name": MODEL_NAME,
        "retrieval_method": "exact_cosine_similarity_sharded",
        "elapsed_seconds": round(time.time() - embedding_start, 2)
    }

print(json.dumps(embedding_stats, ensure_ascii=False, indent=2))

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

{
  "skipped": false,
  "embedding_dir": "/content/semantic_search_full_mpnet_cosine_eval/embedding_shards",
  "metadata_dir": "/content/semantic_search_full_mpnet_cosine_eval/metadata_shards",
  "n_products_embedded": 2410765,
  "n_shards": 97,
  "embedding_dim": 768,
  "model_name": "paraphrase-multilingual-mpnet-base-v2",
  "retrieval_method": "exact_cosine_similarity_sharded",
  "elapsed_seconds": 4709.79
}


In [ ]:
metadata_start = time.time()

if os.path.exists(METADATA_DB_PATH) and not FORCE_REBUILD_METADATA_DB:
    con = duckdb.connect(METADATA_DB_PATH, read_only=True)
    metadata_count = con.execute("SELECT count(*) FROM products").fetchone()[0]
    con.close()
    metadata_stats = {
        "skipped": True,
        "metadata_db_path": METADATA_DB_PATH,
        "metadata_count": int(metadata_count),
        "elapsed_seconds": 0
    }
else:
    if os.path.exists(METADATA_DB_PATH):
        os.remove(METADATA_DB_PATH)

    metadata_glob = os.path.join(METADATA_DIR, "*.parquet")
    con = duckdb.connect(METADATA_DB_PATH)
    con.execute("PRAGMA threads=" + str(max(2, os.cpu_count() or 2)))
    con.execute("CREATE TABLE products AS SELECT * FROM read_parquet(" + sql_literal(metadata_glob) + ")")
    con.execute("CREATE INDEX idx_products_row_id ON products(row_id)")
    con.execute("CREATE INDEX idx_products_category_key ON products(category_key)")
    metadata_count = con.execute("SELECT count(*) FROM products").fetchone()[0]
    con.close()

    metadata_stats = {
        "skipped": False,
        "metadata_db_path": METADATA_DB_PATH,
        "metadata_count": int(metadata_count),
        "elapsed_seconds": round(time.time() - metadata_start, 2)
    }

print(json.dumps(metadata_stats, ensure_ascii=False, indent=2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "skipped": false,
  "metadata_db_path": "/content/semantic_search_full_mpnet_cosine_eval/metadata.duckdb",
  "metadata_count": 2410765,
  "elapsed_seconds": 9.4
}


In [ ]:
embedding_files = sorted(glob.glob(os.path.join(EMBEDDING_DIR, "embeddings_*.npy")))
metadata_files = sorted(glob.glob(os.path.join(METADATA_DIR, "metadata_*.parquet")))

if len(embedding_files) == 0:
    raise ValueError("no embedding shards")

shard_cache = []
for emb_path, meta_path in tqdm(list(zip(embedding_files, metadata_files))):
    row_ids = pq.read_table(meta_path, columns=["row_id"]).column("row_id").to_numpy().astype(np.int64)
    shard_cache.append({
        "emb_path": emb_path,
        "meta_path": meta_path,
        "row_ids": row_ids,
        "n": int(len(row_ids))
    })

metadata_con = duckdb.connect(METADATA_DB_PATH, read_only=True)

def fetch_metadata(row_ids):
    row_ids = [int(x) for x in row_ids]
    if len(row_ids) == 0:
        return pd.DataFrame()
    id_sql = ",".join(str(x) for x in row_ids)
    return metadata_con.execute(
        "SELECT row_id, asin, title, brand, category, category_key, price FROM products WHERE row_id IN (" + id_sql + ")"
    ).fetchdf()

def exact_cosine_search(query, k=10, exclude_row_id=None):
    start = time.time()
    q = model.encode(
        [query],
        batch_size=ENCODE_BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")[0]

    all_scores = []
    all_row_ids = []

    for shard in shard_cache:
        arr = np.load(shard["emb_path"], mmap_mode="r")
        scores = arr @ q
        if exclude_row_id is not None:
            mask = shard["row_ids"] == int(exclude_row_id)
            if mask.any():
                scores = scores.copy()
                scores[mask] = -np.inf
        kk = min(k, len(scores))
        if kk <= 0:
            continue
        idx = np.argpartition(scores, -kk)[-kk:]
        all_scores.append(scores[idx])
        all_row_ids.append(shard["row_ids"][idx])

    scores = np.concatenate(all_scores)
    row_ids = np.concatenate(all_row_ids)
    top = np.argsort(scores)[::-1][:k]
    final_row_ids = row_ids[top].astype(np.int64)
    final_scores = scores[top].astype(float)

    meta = fetch_metadata(final_row_ids.tolist())
    meta_map = {int(r.row_id): r for r in meta.itertuples(index=False)}
    rows = []

    for rank, (row_id, score) in enumerate(zip(final_row_ids, final_scores), start=1):
        if int(row_id) in meta_map:
            r = meta_map[int(row_id)]
            rows.append({
                "rank": rank,
                "score": float(score),
                "row_id": int(row_id),
                "asin": r.asin,
                "title": r.title,
                "brand": r.brand,
                "category": r.category,
                "category_key": r.category_key,
                "price": r.price
            })

    result = pd.DataFrame(rows)
    result.attrs["latency_seconds"] = round(time.time() - start, 4)
    return result

def exact_cosine_search_batch(query_embeddings, k=10, exclude_row_ids=None):
    q = np.asarray(query_embeddings, dtype="float32")
    bsz = q.shape[0]
    candidate_scores = [[] for _ in range(bsz)]
    candidate_row_ids = [[] for _ in range(bsz)]
    use_gpu = torch.cuda.is_available() and USE_GPU_FOR_BATCH_RETRIEVAL

    if use_gpu:
        q_t = torch.from_numpy(q).to(device)
        if USE_FP16_ON_GPU:
            q_t = q_t.half()

    for shard in shard_cache:
        arr_np = np.load(shard["emb_path"], mmap_mode="r")
        row_ids = shard["row_ids"]

        if use_gpu:
            arr_t = torch.from_numpy(np.asarray(arr_np)).to(device)
            if USE_FP16_ON_GPU:
                arr_t = arr_t.half()
            scores_mat = torch.matmul(arr_t, q_t.T).float().cpu().numpy()
            del arr_t
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            scores_mat = arr_np @ q.T

        for qi in range(bsz):
            scores = scores_mat[:, qi]
            if exclude_row_ids is not None:
                ex = int(exclude_row_ids[qi])
                mask = row_ids == ex
                if mask.any():
                    scores = scores.copy()
                    scores[mask] = -np.inf
            kk = min(k, len(scores))
            idx = np.argpartition(scores, -kk)[-kk:]
            candidate_scores[qi].append(scores[idx])
            candidate_row_ids[qi].append(row_ids[idx])

        del scores_mat

    outputs = []
    for qi in range(bsz):
        scores = np.concatenate(candidate_scores[qi])
        row_ids = np.concatenate(candidate_row_ids[qi])
        top = np.argsort(scores)[::-1][:k]
        outputs.append((row_ids[top].astype(np.int64), scores[top].astype(float)))

    return outputs

print(len(shard_cache))

  0%|          | 0/97 [00:00<?, ?it/s]

97


In [ ]:
eval_start = time.time()

if os.path.exists(EVAL_SUMMARY_PATH) and os.path.exists(EVAL_METRICS_PATH) and not FORCE_REBUILD_EVAL:
    eval_df = pd.read_csv(EVAL_METRICS_PATH)
    eval_summary = pd.read_csv(EVAL_SUMMARY_PATH)
else:
    eval_items = metadata_con.execute(f'''
    WITH valid_categories AS (
        SELECT category_key, COUNT(*) AS cnt
        FROM products
        WHERE category_key IS NOT NULL AND LENGTH(category_key) > 2
        GROUP BY category_key
        HAVING COUNT(*) >= {EVAL_K + 1}
    )
    SELECT p.row_id, p.title, p.category, p.category_key
    FROM products p
    JOIN valid_categories c ON p.category_key = c.category_key
    WHERE p.title IS NOT NULL AND LENGTH(p.title) > 10
    ORDER BY random()
    LIMIT {EVAL_QUERY_COUNT}
    ''').fetchdf()

    eval_rows = []

    for start_idx in tqdm(range(0, len(eval_items), EVAL_QUERY_BATCH_SIZE)):
        batch_df = eval_items.iloc[start_idx:start_idx + EVAL_QUERY_BATCH_SIZE].copy()
        queries = batch_df["title"].astype(str).tolist()
        q_embs = model.encode(
            queries,
            batch_size=ENCODE_BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype("float32")

        batch_t0 = time.time()
        search_outputs = exact_cosine_search_batch(
            q_embs,
            k=EVAL_K,
            exclude_row_ids=batch_df["row_id"].astype(int).tolist()
        )
        batch_latency = (time.time() - batch_t0) / max(1, len(batch_df))

        all_result_ids = []
        for row_ids, _ in search_outputs:
            all_result_ids.extend([int(x) for x in row_ids])
        meta = fetch_metadata(sorted(set(all_result_ids)))
        meta_map = {int(r.row_id): r for r in meta.itertuples(index=False)}

        for local_i, (_, item) in enumerate(batch_df.iterrows()):
            target_category = str(item["category_key"])
            row_ids, scores = search_outputs[local_i]
            rels = []
            for row_id in row_ids:
                r = meta_map.get(int(row_id))
                rels.append(1 if r is not None and str(r.category_key) == target_category else 0)

            precision = float(np.mean(rels[:EVAL_K])) if len(rels) else 0.0
            hit = float(1.0 if any(r > 0 for r in rels[:EVAL_K]) else 0.0)
            mrr = mrr_at_k(rels, EVAL_K)
            ndcg = ndcg_at_k(rels, EVAL_K)

            eval_rows.append({
                "query_row_id": int(item["row_id"]),
                "query_title": item["title"],
                "query_category": item["category"],
                "query_category_key": item["category_key"],
                "precision_at_10": precision,
                "hit_at_10": hit,
                "mrr_at_10": mrr,
                "ndcg_at_10": ndcg,
                "latency_seconds": float(batch_latency),
                "relevant_count_at_10": int(sum(rels[:EVAL_K]))
            })

    eval_df = pd.DataFrame(eval_rows)
    eval_summary = pd.DataFrame([{
        "model_name": MODEL_NAME,
        "retrieval_method": "exact_cosine_similarity_same_as_demo_sharded_gpu_batched",
        "ground_truth": "same_category_metadata_proxy",
        "n_eval_queries": int(len(eval_df)),
        "precision_at_10": float(eval_df["precision_at_10"].mean()),
        "hit_at_10": float(eval_df["hit_at_10"].mean()),
        "mrr_at_10": float(eval_df["mrr_at_10"].mean()),
        "ndcg_at_10": float(eval_df["ndcg_at_10"].mean()),
        "mean_latency_seconds": float(eval_df["latency_seconds"].mean()),
        "elapsed_seconds": round(time.time() - eval_start, 2)
    }])

    eval_df.to_csv(EVAL_METRICS_PATH, index=False)
    eval_summary.to_csv(EVAL_SUMMARY_PATH, index=False)

display(eval_summary)
display(eval_df.head(10))

  0%|          | 0/7 [00:00<?, ?it/s]

/tmp/ipykernel_13809/1559416298.py:102: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  arr_t = torch.from_numpy(np.asarray(arr_np)).to(device)


,model_name,retrieval_method,ground_truth,n_eval_queries,precision_at_10,hit_at_10,mrr_at_10,ndcg_at_10,mean_latency_seconds,elapsed_seconds
0,paraphrase-multilingual-mpnet-base-v2,exact_cosine_similarity_same_as_demo_sharded_g...,same_category_metadata_proxy,200,0.782,0.965,0.874929,0.888731,0.117683,24.57


,query_row_id,query_title,query_category,query_category_key,precision_at_10,hit_at_10,mrr_at_10,ndcg_at_10,latency_seconds,relevant_count_at_10
0,678029,IQ Shield Matte Screen Protector Compatible wi...,Cell Phones & Accessories,cell phones accessories,1.0,1.0,1.000000,1.000000,0.110749,10
1,407513,Ringke Cell Phone Case for Google Motorola Nex...,Cell Phones & Accessories,cell phones accessories,1.0,1.0,1.000000,1.000000,0.110749,10
2,951140,Neewer Video Camcorder Camera DC/DV Handle gri...,Camera & Photo,camera photo,0.9,1.0,1.000000,0.950421,0.110749,9
3,900386,13mp Wide Angle USB Camera Module with Microph...,Computers,computers,0.7,1.0,1.000000,0.961921,0.110749,7
4,898378,Replacement Internal Speakers Left Right Fit f...,Computers,computers,0.1,1.0,0.142857,0.333333,0.110749,1
5,371708,"15.6"" WXGA Laptop LCD LED Screen for Dell Insp...",Computers,computers,0.9,1.0,1.000000,0.977015,0.110749,9
6,98682,Caka Marble Case for Galaxy S20 Plus Case for ...,Cell Phones & Accessories,cell phones accessories,1.0,1.0,1.000000,1.000000,0.110749,10
7,2036833,"For Galaxy S3 , ivencase [Be Happy] Personalit...",Cell Phones & Accessories,cell phones accessories,1.0,1.0,1.000000,1.000000,0.110749,10
8,1564444,Amplim Apple iPhone 5S 5 Space Gray Premium Sl...,Cell Phones & Accessories,cell phones accessories,0.9,1.0,1.000000,0.997188,0.110749,9
9,2035488,NewPowerGear USB Printer Scanner Cable Cord fo...,Computers,computers,0.6,1.0,1.000000,0.870715,0.110749,6


In [ ]:
for q in ["laptop học lập trình", "tai nghe chống ồn", "màn hình chơi game 27 inch", "chuột không dây văn phòng", "bàn phím cơ"]:
    r = exact_cosine_search(q, 10)
    print("\nQUERY:", q)
    print("LATENCY:", r.attrs.get("latency_seconds"))
    display(r[["rank", "score", "title", "brand", "category", "price"]])


QUERY: laptop học lập trình
LATENCY: 0.7573


,rank,score,title,brand,category,price
0,1,0.680260,Lenovo - 300W Gen 3-2-in-1 Educational Compute...,Lenovo,Computers,171.0
1,2,0.666479,"Lenovo ThinkPad 11e 11.6"" LED Chromebook Lapto...",Lenovo,Computers,44.88
2,3,0.658849,"Lenovo ThinkPad 11E (5th Gen) 11.6"" HD Busines...",Lenovo,Computers,150.0
3,4,0.657669,"Lenovo ThinkPad 11e Laptop 11.6"", Intel Celero...",Amazon Renewed,Computers,130.0
4,5,0.657287,Dell Chromebook 11 11.6' Notebook - Intel Cele...,Dell,Computers,
5,6,0.656971,Lenovo 14w Gen 2 14” Laptop Student Notebook 2...,Lenovo,Computers,229.0
6,7,0.656379,"Lenovo ThinkPad 11e Laptop 11.6in, Intel Celer...",Amazon Renewed,Computers,
7,8,0.653059,Lenovo X1 Carbon 34442DU 14-Inch Laptop,Lenovo,Computers,
8,9,0.649822,"HP Stream 14-inch Laptop, Intel Celeron N4000 ...",HP,Computers,
9,10,0.647225,"Dell Latitude LAT3160-1333BLK 11.6"" HD Touchsc...",Dell,Computers,



QUERY: tai nghe chống ồn
LATENCY: 0.6475


,rank,score,title,brand,category,price
0,1,0.786628,NC300 Noise Cancelling Headphones,Memorex,Home Audio & Theater,
1,2,0.764547,Sony MDRZX110NA Analog Noise Cancelling Headph...,Sony,Home Audio & Theater,
2,3,0.759232,Memorex NC300 Noise Cancellation On-Ear Headph...,Memorex,Home Audio & Theater,
3,4,0.749759,"EMPERSTAR Noise Cancelling Headphones, Over Th...",EMPERSTAR,All Electronics,39.0
4,5,0.749118,Wicked Audio WI1852 In-Ear Deuce Earbuds,Wicked Audio,Home Audio & Theater,
5,6,0.748617,Active Noise Cancelling Earbuds AMZLIFE Blueto...,AMZLIFE,Home Audio & Theater,
6,7,0.747851,Wicked Audio WI1855 In-Ear Deuce Earbuds,Wicked Audio,Home Audio & Theater,19.99
7,8,0.747627,Wicked Audio WI1857 IN-EAR DEUCE EARbuds,Wicked Audio,Home Audio & Theater,
8,9,0.747351,Noise Isolating in-Ear Earbuds for Stereo Soun...,ANKIT,Home Audio & Theater,
9,10,0.747068,Avantree ANC032 Active Noise Cancelling Headph...,Avantree,All Electronics,



QUERY: màn hình chơi game 27 inch
LATENCY: 0.6647


,rank,score,title,brand,category,price
0,1,0.694801,"LG Electronics Monitor 27MC67-B 27"" Screen LCD...",LG,Computers,
1,2,0.694090,"AOC 27G2E 27"" 16:9 Full HD 144Hz IPS Gaming Mo...",AOC,All Electronics,
2,3,0.683340,FPS/RTS Optimized VIOTEK GN27C 27” Curved Comp...,Viotek,Computers,
3,4,0.680705,HP X27i 27” 2k Gaming Monitor with AMD FreeSyn...,HP,Computers,
4,5,0.671967,SAMSUNG CFG70 Series 27-Inch 1ms Curved Gaming...,SAMSUNG,Computers,
5,6,0.670369,LG 25UM58-P UltraWide Monitor 25'' 21:9 FHD ()...,LG,Computers,
6,7,0.666176,LG 27GN750-B UltraGear Gaming Monitor 27” FHD ...,LG,Computers,
7,8,0.665195,SAMSUNG 27-Inch Wide Viewing Angle LED Monitor...,SAMSUNG,Computers,
8,9,0.662008,ViewSync New VSG27TF-165K Real 165Hz Gaming 27...,ViewSync,Computers,
9,10,0.658444,"2K Portable Monitor 17.3"", 2560x1440 Compatibl...",MSDONG,Computers,



QUERY: chuột không dây văn phòng
LATENCY: 0.7858


,rank,score,title,brand,category,price
0,1,0.723930,Mad Catz Office RAT m Wireless Bluetooth Optic...,Mad Catz,Computers,
1,2,0.719180,Mad Catz Office RAT m Wireless Bluetooth Optic...,Mad Catz,Computers,
2,3,0.709929,"Wireless Mouse, 2.4G Slim Portable Cordless Mo...",Generic,All Electronics,
3,4,0.707207,Microsoft D5D-00038 Wireless Mobile Mouse 4000...,Microsoft,Computers,33.99
4,5,0.695958,Microsoft Wireless Mobile Mouse 4000 Studio Se...,Microsoft,All Electronics,99.99
5,6,0.692924,Microsoft Software-Microsoft Wireless Mobile M...,Microsoft,Computers,
6,7,0.691215,Wireless Mouse - Brick-reddish with Black Stripes,Road Mice,All Electronics,
7,8,0.689544,Wireless Mouse - Camaro Highway Patrol,Road Mice,All Electronics,
8,9,0.688669,Wireless Mouse - Dodge Charger Police,Road Mice,All Electronics,
9,10,0.686150,Wired Mouse - Dodge Charger Police,Road Mice,All Electronics,



QUERY: bàn phím cơ
LATENCY: 0.9352


,rank,score,title,brand,category,price
0,1,0.791326,"AZERTY French Keyboard Stickers (Orange, See-t...",Latkey,Computers,
1,2,0.782820,"Dilter Wired Keyboard, 104 Keys Full-Sized Typ...",Dilter,Computers,29.99
2,3,0.781458,"Gaming Keyboard Chocolate, STOGA Typewriter Ke...",STOGA,Computers,29.99
3,4,0.772600,Taeeiancd Typewriter Keyboard 104-key Retro Pu...,Taeeiancd,Computers,20.99
4,5,0.771627,"WREWING Wired Keyboard, LED Backlight Full Siz...",WREWING,Computers,
5,6,0.769128,"Adesso Akb150eb Tru-Form Wired Keyboard, Black...",Adesso,Computers,44.93
6,7,0.768338,"Mechanical Keyboard, Webats Optical Axis Gamin...",Webat,Computers,
7,8,0.768043,SafeType Keyboard - Black Color V902,Safetype,Computers,
8,9,0.764623,"Gaming Keyboard , Punk Typewriter Style keyboa...",Vbestlife,Computers,
9,10,0.762773,DANSHER Percent 60% Mechanical Gaming Keyboard...,DANSHER,Computers,36.99


In [ ]:
run_stats = {
    "input_path": INPUT_PATH,
    "work_dir": WORK_DIR,
    "export_dir": EXPORT_DIR,
    "model_name": MODEL_NAME,
    "retrieval_method": "exact_cosine_similarity_same_as_demo",
    "etl": etl_stats,
    "embedding": embedding_stats,
    "metadata": metadata_stats
}

with open(RUN_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(run_stats, f, ensure_ascii=False, indent=2)

if SAVE_TO_DRIVE:
    os.makedirs(EXPORT_DIR, exist_ok=True)
    export_items = [
        (CLEAN_PARQUET_PATH, "clean_products.parquet"),
        (METADATA_DB_PATH, "metadata.duckdb"),
        (ETL_STATS_PATH, "etl_stats.json"),
        (RUN_STATS_PATH, "run_stats.json"),
        (EVAL_METRICS_PATH, "category_evaluation_metrics.csv"),
        (EVAL_SUMMARY_PATH, "category_evaluation_summary.csv")
    ]

    for src, name in export_items:
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(EXPORT_DIR, name))

    for folder, name in [(EMBEDDING_DIR, "embedding_shards"), (METADATA_DIR, "metadata_shards")]:
        dst = os.path.join(EXPORT_DIR, name)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(folder, dst)

print(json.dumps(run_stats, ensure_ascii=False, indent=2))
print(EXPORT_DIR)
print(os.listdir(EXPORT_DIR)[:20])

{
  "input_path": "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/san_pham.parquet",
  "work_dir": "/content/semantic_search_full_mpnet_cosine_eval",
  "export_dir": "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/semantic_search_full_mpnet_cosine_eval",
  "model_name": "paraphrase-multilingual-mpnet-base-v2",
  "retrieval_method": "exact_cosine_similarity_same_as_demo",
  "etl": {
    "skipped": false,
    "input_path": "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/san_pham.parquet",
    "clean_parquet_path": "/content/semantic_search_full_mpnet_cosine_eval/clean_products.parquet",
    "raw_count": 2410915,
    "clean_count": 2410765,
    "column_map": {
      "asin": "asin",
      "title": "title",
      "brand": "store",
      "category": "main_category",
      "description": "description",
      "features": "features",
      "details": "details",
      "price": "price"
    },
    "elap